In [0]:
 
from pyspark.sql import functions as F
 
CATALOGO, ESQUEMA = "workspace", "horoz"
T = lambda nombre: f"{CATALOGO}.{ESQUEMA}.{nombre}"
KEY = ["LocationId", "BusinessDate", "Id"]
 
resumen = {}   # se llena en cada sección y se muestra al final
 
 
def aviso(seccion, e):
    print(f"⚠️ La sección {seccion} no se pudo calcular: {str(e)[:300]}")
 

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 · Calidad de datos
# MAGIC
# MAGIC Evidencia de cómo el pipeline `horoz_fraude_pipeline` detecta y maneja problemas de calidad.
# MAGIC
# MAGIC | Sección | Pregunta |
# MAGIC |---|---|
# MAGIC | 1 | ¿Cuántos registros hay en cada capa? |
# MAGIC | 2 | ¿Cuántos duplicados y llaves incompletas había? |
# MAGIC | 3 | Los tres tipos de expectations: ALLOW, DROP y FAIL |
# MAGIC | 4 | ¿Qué reporta el event log del pipeline? |
# MAGIC | 5 | ¿Qué columnas tienen valores nulos? |
# MAGIC | 6 | ¿Llegaron campos fuera del esquema (`_rescued_data`)? |
# MAGIC | 7 | ¿Qué formatos inconsistentes se normalizaron? |
# MAGIC | 8 | ¿Qué valores atípicos quedan y qué significan? |
# MAGIC | 9 | Resumen para el reporte |
# MAGIC
# MAGIC **Requisito:** el pipeline debe haber terminado sin errores. Conectar a cómputo **Serverless**.
# MAGIC Cada sección está protegida: si una falla, imprime el error y **Run all** sigue con las demás.

# COMMAND ----------

from pyspark.sql import functions as F

CATALOGO, ESQUEMA = "workspace", "horoz"
T = lambda nombre: f"{CATALOGO}.{ESQUEMA}.{nombre}"
KEY = ["LocationId", "BusinessDate", "Id"]

resumen = {}   # se llena en cada sección y se muestra al final


def aviso(seccion, e):
    print(f"⚠️ La sección {seccion} no se pudo calcular: {str(e)[:300]}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Registros por capa

# COMMAND ----------

tablas = ["bronze_checks", "bronze_employees", "bronze_items", "bronze_locations", "bronze_revenue_centers",
          "silver_checks", "silver_items", "silver_voids", "silver_discounts", "dq_gate_silver",
          "gold_daily_sales", "gold_employee_activity", "gold_employee_weekly_risk", "gold_checks_to_review"]

conteos = []
for t in tablas:
    capa = "control" if t.startswith("dq_") else t.split("_")[0]
    try:
        conteos.append((capa, t, spark.table(T(t)).count()))
    except Exception as e:
        conteos.append((capa, t, None))
        print(f"No se pudo leer {t}: {str(e)[:150]}")

display(spark.createDataFrame(conteos, "capa string, tabla string, registros long"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Duplicados y llaves incompletas
# MAGIC La llave de un ticket es `LocationId + BusinessDate + Id`. `checks_dedup` quita copias con
# MAGIC `dropDuplicates` y la expectation `llave_completa` (DROP) elimina tickets sin llave.
# MAGIC La fuente no trae duplicados (los 16,491 del EDA venían de nuestra propia carga), así que el
# MAGIC `dropDuplicates` queda como **protección** por si en el futuro llega un archivo repetido.

# COMMAND ----------

bronze = spark.table(T("bronze_checks"))

n_bronze     = bronze.count()
n_llave_nula = bronze.filter("Id IS NULL OR LocationId IS NULL OR BusinessDate IS NULL").count()
n_distintos  = bronze.select(*KEY).distinct().count()
n_silver     = spark.table(T("silver_checks")).count()
n_duplicados = n_bronze - n_distintos

print(f"Tickets en bronze:                 {n_bronze:>12,}")
print(f"Copias duplicadas (misma llave):   {n_duplicados:>12,}")
print(f"Tickets con llave incompleta:      {n_llave_nula:>12,}")
print(f"Tickets en silver_checks:          {n_silver:>12,}")
print(f"Eliminados en total:               {n_bronze - n_silver:>12,}")

resumen["duplicados"] = n_duplicados
resumen["llave_nula"] = n_llave_nula

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Los tres tipos de expectations
# MAGIC Se recalcula cada regla directo sobre las tablas, con su tipo de acción:
# MAGIC
# MAGIC | Acción | Cuándo se usa | Qué pasa con el registro |
# MAGIC |---|---|---|
# MAGIC | **ALLOW** (`expect`) | Anomalías que son evidencia de fraude o de captura | Se conserva y se registra en métricas |
# MAGIC | **DROP** (`expect_or_drop`) | Registros inservibles (sin llave) | Se elimina |
# MAGIC | **FAIL** (`expect_or_fail`) | Fallas sistémicas: si se rompen, todo lo de abajo saldría mal | Se detiene la actualización completa |

# COMMAND ----------

reglas = [
    # (tabla, expectation, acción, condición)
    ("checks_dedup",  "llave_completa",              "DROP",  "Id IS NOT NULL AND LocationId IS NOT NULL AND BusinessDate IS NOT NULL"),
    ("silver_checks", "horas_validas",               "ALLOW", "close_ts >= open_ts"),
    ("silver_checks", "pago_cuadra",                 "ALLOW", "abs(dif_pago) <= 0.05"),
    ("silver_checks", "comensales_en_comedor",       "ALLOW", "RevenueCenterId <> 1 OR TrafficCount > 0"),
    ("silver_checks", "un_desc_empleado_por_ticket", "ALLOW", "n_desc_empleado <= 1"),
    ("silver_checks", "desc_empleado_no_combinado",  "ALLOW", "n_desc_empleado = 0 OR n_desc_otros = 0"),
    ("silver_checks", "void_no_combinado_con_desc",  "ALLOW", "n_voids = 0 OR n_desc_total = 0"),
    ("silver_items",  "precio_no_negativo",          "ALLOW", "SoldPrice >= 0"),
    ("dq_gate_silver", "hay_tickets",                "FAIL",  "tickets > 0"),
    ("dq_gate_silver", "llave_unica",                "FAIL",  "tickets = llaves_distintas"),
    ("dq_gate_silver", "estructura_pagos_valida",    "FAIL",  "pct_no_cuadra <= 1.0"),
]

# checks_dedup es una vista temporal del pipeline: su regla se evalúa sobre bronze_checks
fuente = {"checks_dedup": "bronze_checks"}

filas = []
for tabla in dict.fromkeys(r[0] for r in reglas):
    rs = [r for r in reglas if r[0] == tabla]
    try:
        df = spark.table(T(fuente.get(tabla, tabla)))
        aggs = [F.count("*").alias("_total")]
        for _, n, _, c in rs:
            aggs += [F.sum(F.when(F.expr(c) == F.lit(False), 1).otherwise(0)).alias(f"{n}__falla"),
                     F.sum(F.when(F.expr(c).isNull(), 1).otherwise(0)).alias(f"{n}__nulo")]
        r = df.agg(*aggs).first().asDict()
        total = r["_total"]
        for _, n, accion, c in rs:
            falla = r[f"{n}__falla"]
            filas.append((tabla, n, accion, c, total, falla, r[f"{n}__nulo"],
                          round(100 * falla / total, 3) if total else None))
            resumen[n] = falla
    except Exception as e:
        aviso(f"3 ({tabla})", e)

expectations_df = spark.createDataFrame(filas,
    "tabla string, expectation string, accion string, condicion string, registros long, "
    "fallan long, no_evaluables_por_nulo long, pct_falla double")
display(expectations_df.orderBy(F.expr("CASE accion WHEN 'FAIL' THEN 1 WHEN 'DROP' THEN 2 ELSE 3 END"),
                                F.desc("fallan")))

# COMMAND ----------

# Valores reales de la compuerta FAIL en la última corrida
try:
    display(spark.table(T("dq_gate_silver")))
except Exception as e:
    aviso("3 (dq_gate_silver)", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Event log del pipeline
# MAGIC El pipeline publica su registro de eventos como tabla en el esquema. De ahí salen el historial
# MAGIC de corridas (incluida la que falló por memoria) y las métricas de expectations que reporta Databricks.

# COMMAND ----------

try:
    nombre_log = (spark.table(f"{CATALOGO}.information_schema.tables")
                  .filter(f"table_schema = '{ESQUEMA}' AND table_name LIKE 'event_log_%'")
                  .select("table_name").first())
    EVENT_LOG = T(nombre_log.table_name) if nombre_log else f"event_log(TABLE({T('silver_checks')}))"
    print("Leyendo el event log de:", EVENT_LOG)

    historial = spark.sql(f"""
        SELECT origin.update_id                                         AS update_id,
               MIN(timestamp)                                           AS inicio,
               MAX(timestamp)                                           AS fin,
               MAX_BY(details:update_progress:state, timestamp)         AS estado_final
        FROM {EVENT_LOG}
        WHERE event_type = 'update_progress'
        GROUP BY origin.update_id
        ORDER BY inicio DESC
    """)
    print("Historial de corridas (incluye las fallidas):")
    display(historial)
except Exception as e:
    EVENT_LOG = None
    aviso("4 (historial)", e)

# COMMAND ----------

# Errores registrados (por ejemplo, el OutOfMemory de bronze_checks)
try:
    display(spark.sql(f"""
        SELECT timestamp, origin.flow_name AS flujo, level, LEFT(message, 200) AS mensaje
        FROM {EVENT_LOG}
        WHERE level = 'ERROR'
        ORDER BY timestamp DESC
        LIMIT 20
    """))
except Exception as e:
    aviso("4 (errores)", e)

# COMMAND ----------

# Expectations según Databricks, por corrida
try:
    display(spark.sql(f"""
        SELECT update_id, MIN(ts) AS corrida, e.dataset, e.name AS expectation,
               SUM(e.passed_records) AS pasaron, SUM(e.failed_records) AS fallaron
        FROM (
          SELECT origin.update_id AS update_id, timestamp AS ts,
                 explode(from_json(details:flow_progress:data_quality:expectations,
                   'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>')) AS e
          FROM {EVENT_LOG}
          WHERE event_type = 'flow_progress'
            AND details:flow_progress:data_quality:expectations IS NOT NULL
        )
        GROUP BY update_id, e.dataset, e.name
        ORDER BY corrida DESC, fallaron DESC
    """))
except Exception as e:
    aviso("4 (expectations)", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Valores nulos por columna
# MAGIC Solo se muestran columnas con al menos un nulo. En `silver_checks` los montos se calculan con
# MAGIC `coalesce(..., 0)`, así que un nulo en el JSON original se convierte en 0 y no rompe las sumas.

# COMMAND ----------

def nulos(tabla):
    df = spark.table(T(tabla))
    cols = [c for c, t in df.dtypes if not t.startswith(("array", "struct", "map"))]
    fila = df.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in cols]
                     + [F.count("*").alias("__total")]).first().asDict()
    total = fila.pop("__total")
    return [(tabla, c, v, round(100 * v / total, 2)) for c, v in fila.items() if v]


filas_nulos = []
for t in ["silver_checks", "silver_items", "silver_voids", "silver_discounts"]:
    try:
        filas_nulos += nulos(t)
    except Exception as e:
        aviso(f"5 ({t})", e)

if filas_nulos:
    display(spark.createDataFrame(filas_nulos, "tabla string, columna string, nulos long, pct double")
            .orderBy("tabla", F.desc("nulos")))
else:
    print("Ninguna columna simple tiene nulos.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Campos fuera del esquema (`_rescued_data`)
# MAGIC Auto Loader guarda aquí cualquier campo nuevo o con tipo distinto al esperado, en vez de perderlo.

# COMMAND ----------

if "_rescued_data" in bronze.columns:
    rescatados = bronze.filter("_rescued_data IS NOT NULL")
    n_rescatados = rescatados.count()
    print(f"Tickets con datos rescatados: {n_rescatados:,}")
    resumen["rescued_data"] = n_rescatados
    if n_rescatados:
        display(rescatados.select(*KEY, "_rescued_data", "source_file").limit(20))
else:
    print("bronze_checks no tiene columna _rescued_data.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Formatos inconsistentes normalizados
# MAGIC ### 7a. Categorías de void: valor crudo → valor normalizado

# COMMAND ----------

try:
    voids_crudos = (bronze.select(F.explode("Voids").alias("v"))
                    .select(F.col("v.VoidCategoryName").alias("crudo"))
                    .withColumn("normalizado", F.upper(F.trim("crudo")))
                    .withColumn("normalizado",
                                F.when(F.col("normalizado") == "SERV ERR", "SERVER ERR")
                                 .when(F.col("normalizado") == "NOT RECEVD", "NOT RECVD")
                                 .otherwise(F.col("normalizado"))))

    print("Categorías distintas antes:", voids_crudos.select("crudo").distinct().count())
    print("Categorías distintas después:", voids_crudos.select("normalizado").distinct().count())
    display(voids_crudos.groupBy("crudo", "normalizado").count()
            .withColumn("cambio", F.col("crudo") != F.col("normalizado"))
            .orderBy("normalizado", F.desc("count")))
except Exception as e:
    aviso("7a", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7b. Nombres de descuento y su tipo
# MAGIC `promocion` es la categoría por defecto: revisar que no esconda descuentos de empleado o gerente mal escritos.

# COMMAND ----------

try:
    desc = spark.table(T("silver_discounts"))
    crudos = bronze.select(F.explode("Discounts").alias("d")).select(F.col("d.DiscountName").alias("n"))
    print("Nombres distintos antes:", crudos.distinct().count())
    print("Nombres distintos después (upper + trim):", desc.select("DiscountName").distinct().count())

    display(desc.groupBy("tipo_descuento")
            .agg(F.count("*").alias("descuentos"),
                 F.round(F.sum("Amount"), 2).alias("monto"),
                 F.countDistinct("DiscountName").alias("nombres_distintos"))
            .orderBy(F.desc("descuentos")))
except Exception as e:
    aviso("7b", e)

# COMMAND ----------

try:
    display(desc.filter("tipo_descuento = 'promocion'")
            .groupBy("DiscountName").agg(F.count("*").alias("descuentos"), F.round(F.sum("Amount"), 2).alias("monto"))
            .orderBy(F.desc("descuentos")).limit(30))
except Exception as e:
    aviso("7b (promociones)", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 7c. Fechas y horas que no se pudieron convertir

# COMMAND ----------

chk = spark.table(T("silver_checks"))
try:
    display(chk.agg(
        F.sum((F.col("OpenTime").isNotNull() & F.col("open_ts").isNull()).cast("int")).alias("open_no_convertible"),
        F.sum((F.col("CloseTime").isNotNull() & F.col("close_ts").isNull()).cast("int")).alias("close_no_convertible"),
        F.sum(F.col("BusinessDate").isNull().cast("int")).alias("business_date_nula")))
except Exception as e:
    aviso("7c", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Valores atípicos que se conservan

# COMMAND ----------

try:
    display(chk.agg(
        F.sum((F.col("venta") <= 0).cast("int")).alias("tickets_venta_cero_o_negativa"),
        F.sum((F.col("min_mesa") < 0).cast("int")).alias("min_mesa_negativo"),
        F.sum((F.col("min_mesa") > 480).cast("int")).alias("min_mesa_mas_de_8h"),
        F.sum(((F.col("RevenueCenterId") == 1) & (F.col("TrafficCount") > 20)).cast("int")).alias("comedor_mas_de_20_comensales"),
        F.expr("percentile_approx(min_mesa, array(0.5, 0.95, 0.99))").alias("min_mesa_p50_p95_p99")))
except Exception as e:
    aviso("8", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8a. Precios negativos: ¿son errores o ajustes legítimos?
# MAGIC Si los tickets donde aparecen cuadran al centavo, son ajustes registrados a propósito en el POS.

# COMMAND ----------

try:
    neg = spark.sql(f"""
        WITH neg AS (
          SELECT LocationId, BusinessDate, Id, COUNT(*) AS productos, SUM(SoldPrice) AS monto
          FROM {T('silver_items')} WHERE SoldPrice < 0
          GROUP BY LocationId, BusinessDate, Id)
        SELECT COUNT(*)                                  AS tickets_con_precio_negativo,
               SUM(n.productos)                          AS productos_negativos,
               ROUND(SUM(n.monto), 2)                    AS monto_negativo_total,
               SUM(CAST(c.pago_cuadra AS INT))           AS tickets_que_cuadran,
               ROUND(100 * AVG(CAST(c.pago_cuadra AS INT)), 2) AS pct_cuadran
        FROM neg n JOIN {T('silver_checks')} c USING (LocationId, BusinessDate, Id)
    """)
    display(neg)
    resumen["neg_pct_cuadran"] = neg.first().pct_cuadran

    display(spark.table(T("silver_items")).filter("SoldPrice < 0")
            .groupBy("ItemName").agg(F.count("*").alias("veces"), F.round(F.sum("SoldPrice"), 2).alias("monto"))
            .orderBy(F.desc("veces")).limit(20))
except Exception as e:
    aviso("8a", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8b. Comedor sin comensales: ¿problema de una sucursal o de todas?

# COMMAND ----------

try:
    loc = spark.table(T("bronze_locations")).select(F.col("Id").alias("LocationId"), F.col("Name").alias("sucursal"))
    display(chk.filter("RevenueCenterId = 1")
            .groupBy("LocationId")
            .agg(F.count("*").alias("tickets_comedor"),
                 F.sum((F.coalesce("TrafficCount", F.lit(0)) <= 0).cast("int")).alias("sin_comensales"))
            .withColumn("pct_sin_comensales", F.round(100 * F.col("sin_comensales") / F.col("tickets_comedor"), 1))
            .join(loc, "LocationId", "left")
            .orderBy(F.desc("pct_sin_comensales")).limit(20))
except Exception as e:
    aviso("8b", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Resumen para el reporte

# COMMAND ----------

def g(k):
    return resumen.get(k)

neg_txt = (f"ALLOW en silver_items; {g('neg_pct_cuadran')}% de sus tickets cuadran: son ajustes legítimos"
           if g("neg_pct_cuadran") is not None else "ALLOW en silver_items")

filas_resumen = [
    ("FAIL",  "Tickets en silver (hay_tickets)",       g("hay_tickets"),                 "Compuerta dq_gate_silver: si no hay tickets, se detiene todo"),
    ("FAIL",  "Llave repetida en silver (llave_unica)", g("llave_unica"),                 "Compuerta dq_gate_silver: si la llave se repite, se detiene todo"),
    ("FAIL",  "Más del 1% no cuadra",                   g("estructura_pagos_valida"),     "Compuerta dq_gate_silver: indicaría un cambio en la estructura de pagos"),
    ("DROP",  "Llave incompleta",                       g("llave_nula"),                  "expect_or_drop llave_completa"),
    ("—",     "Duplicados exactos",                     g("duplicados"),                  "La fuente no trae duplicados; dropDuplicates queda como protección"),
    ("ALLOW", "Comedor sin comensales",                 g("comensales_en_comedor"),       "Se conserva; las métricas por comensal solo usan TrafficCount > 0"),
    ("ALLOW", "Void + descuento en el mismo ticket",    g("void_no_combinado_con_desc"),  "Se conserva; cuenta como violación en el risk score"),
    ("ALLOW", "Descuento de empleado combinado",        g("desc_empleado_no_combinado"),  "Se conserva; cuenta como violación en el risk score"),
    ("ALLOW", "Más de un descuento de empleado",        g("un_desc_empleado_por_ticket"), "Se conserva; cuenta como violación en el risk score"),
    ("ALLOW", "Pago no cuadra",                         g("pago_cuadra"),                 "Se conserva; aislados en gold_checks_to_review"),
    ("ALLOW", "Hora de cierre antes de apertura",       g("horas_validas"),               "Se conserva"),
    ("ALLOW", "Precio de producto negativo",            g("precio_no_negativo"),          neg_txt),
    ("—",     "Campos fuera de esquema",                g("rescued_data"),                "Capturados en _rescued_data por Auto Loader"),
]
display(spark.createDataFrame(filas_resumen, "accion string, problema string, registros long, manejo string"))